# Merge Sort

Merge sort is a divide-and-conquer algorithm that divides the array into halves, sorts them recursively, and then merges the sorted halves.

## Algorithm Properties

- **Time Complexity:** O(n log n) in all cases
- **Space Complexity:** O(n) auxiliary space
- **Stable:** Yes (maintains relative order of equal elements)
- **In-place:** No (requires additional space)

## Key Operations

1. **Merge two sorted lists**
2. **Merge subarrays** 
3. **Recursive merge sort**

## Time Complexity Analysis

**Recurrence:** T(n) = 2T(n/2) + Θ(n), T(1) = Θ(1)

**Recursion tree solution:**
- Level 0: n work
- Level 1: 2 × n/2 = n work
- Level k: 2^k × n/2^k = n work
- Height: n/2^k = 1 → k = log₂n
- **Total: n × log₂n = Θ(n log n)**

At each level of recursion:
- Split input into 2 parts: O(1)
- Merge sorted parts: Θ(n)
- Height of recursion tree: O(log n)

**Total:** O(n × log n)

![Merge Sort Divide and Merge](images/merge-sort-divide-merge.png)

# Merge Two Sorted Lists

The primitive the whole algorithm is built on. Because both inputs are sorted, the next
smallest element overall can only be at the front of one of them - so compare the two
heads, take the smaller, advance that pointer.

```
a = [10, 15, 20]   b = [5, 6, 6, 30]

i=0 j=0   10 vs 5  → take 5     res [5]
i=0 j=1   10 vs 6  → take 6     res [5,6]
i=0 j=2   10 vs 6  → take 6     res [5,6,6]
i=0 j=3   10 vs 30 → take 10    res [5,6,6,10]
i=1 j=3   15 vs 30 → take 15    res [5,6,6,10,15]
i=2 j=3   20 vs 30 → take 20    res [5,6,6,10,15,20]
a exhausted → append the rest of b → [5,6,6,10,15,20,30]
```

One of the two lists always runs out first, so the two `extend` calls at the end flush
whatever remains. Skipping them silently drops elements - a common bug.

`merge_naive` above shows what this buys: concatenating and re-sorting throws away the
sortedness and pays O((m+n) log(m+n)) for information it already had.

**Time:** Θ(m + n) &nbsp; **Space:** Θ(m + n) for the result

In [ ]:
def merge_naive(a, b):
    """
    Naive approach: concatenate and sort
    Time complexity: O((m+n) * log(m+n))
    Doesn't use the fact that both lists are sorted
    """
    res = a + b
    res.sort()
    return res

def merge_lists(a, b):
    """
    Efficient approach using two pointers
    Time Complexity: Θ(m+n)
    """
    res = []
    m, n = len(a), len(b)
    i = j = 0
    while i < m and j < n:
        if a[i] < b[j]:
            res.append(a[i])
            i += 1
        else:
            res.append(b[j])
            j += 1
    res.extend(a[i:])
    res.extend(b[j:])
    return res

def test_merge_lists():
    result = merge_lists([10, 15, 20], [5, 6, 6, 30])
    assert result == [5, 6, 6, 10, 15, 20, 30]
    
    # Test empty lists
    assert merge_lists([], [1, 2, 3]) == [1, 2, 3]
    assert merge_lists([1, 2, 3], []) == [1, 2, 3]

test_merge_lists()

# Merge Subarrays

The same merge, adapted to work on *one* array. Instead of two lists there are two sorted
ranges sitting side by side - `a[low..mid]` and `a[mid+1..high]` - and the result has to
land back in `a[low..high]`.

The complication: writing into `a` would overwrite elements not yet read. So the two
halves are **copied out** first, then merged back in over the original range with the
write cursor `k`.

```
a = [10, 15, 20, 11, 13]   low=0 mid=2 high=4

left  = [10, 15, 20]     right = [11, 13]

k=0  10 vs 11 → a[0]=10
k=1  15 vs 11 → a[1]=11
k=2  15 vs 13 → a[2]=13
k=3  right exhausted → copy 15, 20 → a[3]=15, a[4]=20

a = [10, 11, 13, 15, 20]
```

`left[i] <= right[j]` (rather than `<`) is what keeps the merge stable: on a tie the
element from the left half goes first, preserving the original order.

This copying is also the reason merge sort is not in-place - it is the O(n) auxiliary
space in the complexity table.

**Time:** Θ(high - low) &nbsp; **Space:** Θ(high - low)

In [ ]:
def merge(a, low, mid, high):
    """
    Merge two sorted subarrays in-place
    Left subarray: a[low...mid]
    Right subarray: a[mid+1...high]
    """
    left = a[low:mid + 1]
    right = a[mid + 1:high + 1]
    
    i = j = 0
    k = low
    
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:  # <= ensures stable merge
            a[k] = left[i]
            i += 1
        else:
            a[k] = right[j]
            j += 1
        k += 1
    
    # Copy remaining elements
    while i < len(left):
        a[k] = left[i]
        i += 1
        k += 1
    
    while j < len(right):
        a[k] = right[j]
        j += 1
        k += 1

def test_merge():
    a = [10, 15, 20, 11, 13]
    merge(a, 0, 2, 4)
    assert a == [10, 11, 13, 15, 20]
    
    a = [5, 8, 12, 14, 7]
    merge(a, 0, 3, 4)
    assert a == [5, 7, 8, 12, 14]

test_merge()

# Merge Sort Algorithm

Divide and conquer in its purest form: a one-element array is already sorted, so split until
you reach that base case, then merge on the way back up. The diagram at the top of the
notebook traces both halves of that process.

Note where the work actually happens - the split is trivial arithmetic
(`m = (l + r) // 2`), and everything is accomplished by the merges during the unwind. The
`if r > l` guard is the base case: a range of one element or none needs no work.

Each level of recursion merges Θ(n) elements in total and there are log n levels. Unlike
quick sort, no input can unbalance the split, so that O(n log n) holds in *every* case.

**Time:** Θ(n log n) always &nbsp; **Space:** O(n) auxiliary + O(log n) stack

In [ ]:
def merge_sort(a, l, r):
    """
    Recursive merge sort
    a: array to sort
    l: left index
    r: right index
    """
    if r > l:  # At least 2 elements needed
        m = (r + l) // 2
        merge_sort(a, l, m)      # Sort left half
        merge_sort(a, m + 1, r)  # Sort right half
        merge(a, l, m, r)        # Merge sorted halves

def test_merge_sort():
    a = [10, 5, 30, 15, 7]
    merge_sort(a, 0, len(a) - 1)
    assert a == [5, 7, 10, 15, 30]
    
    # Test edge cases
    a = [1]
    merge_sort(a, 0, 0)
    assert a == [1]
    
    a = [3, 1, 4, 1, 5, 9, 2, 6]
    merge_sort(a, 0, len(a) - 1)
    assert a == [1, 1, 2, 3, 4, 5, 6, 9]

test_merge_sort()